In [1]:
import pandas as pd
import numpy as np
from scipy.io import savemat

In [2]:
choice = 6  #start from 0
devices = ['nfet_03v3', 'pfet_03v3','nfet_05v0', 'pfet_05v0','nfet_06v', 'pfet_06v0', 'nfet_10v0']

# widths used for characterization
w = np.array([5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0])
nfing = np.array([1, 1, 1, 1, 1, 1, 1])
Lmin = np.array([0.28, 0.28, 0.6, 0.5, 0.6, 0.6, 0.6])

In [3]:
# read ngspice data
df_raw = pd.read_csv('./simulation/techsweep_'+devices[choice]+'.txt', sep='\s+')
par_names = df_raw.columns.to_list()
fet_name = par_names[1].split('[')[0]

# remove unwanted columns and rename for readability
df = df_raw.drop(['frequency', 'frequency.1'], axis=1)
df = df.apply(pd.to_numeric)
df.columns = df.columns.str.replace(fet_name, '')
df.columns = df.columns.str.replace(fet_name[1:], '')
df.columns = df.columns.str.replace('[dc]', '')
df.columns = df.columns.str.replace('onoise..', 'n')
df.columns = df.columns.str.removeprefix('@')
df.columns = df.columns.str.removeprefix('[')
df.columns = df.columns.str.removesuffix(']')
df

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_734/3721318769.py:2: SyntaxWarning: invalid escape sequence '\s'
  df_raw = pd.read_csv('./simulation/techsweep_'+devices[choice]+'.txt', sep='\s+')


,capbd,capbs,cdd,cgb,cgd,cgdo,cgg,cgs,cgso,css,...,gm,gmbs,id,l,vth,vb,vd,vg,n1overf,nid
0,8.742000e-15,3.750000e-15,3.553000e-17,-1.431000e-15,-1.338000e-17,3.724000e-16,1.438000e-15,6.012000e-18,3.724000e-16,1.710000e-17,...,2.330000e-41,1.421000e-41,9.498000e-43,6.000000e-07,0.9351,0.0,0.000,0.0,2.973000e-46,1.351000e-17
1,8.070000e-15,3.559000e-15,1.594000e-17,-1.303000e-15,-6.096000e-18,3.724000e-16,1.307000e-15,2.379000e-18,3.724000e-16,7.599000e-18,...,2.396000e-24,1.251000e-24,9.509000e-26,6.000000e-07,1.0250,-0.2,0.000,0.0,3.088000e-29,3.321000e-18
2,7.607000e-15,3.421000e-15,7.798000e-18,-1.204000e-15,-3.023000e-18,3.724000e-16,1.206000e-15,9.763000e-19,3.724000e-16,3.695000e-18,...,4.056000e-25,1.860000e-25,1.578000e-26,6.000000e-07,1.1040,-0.4,0.000,0.0,5.265000e-30,9.549000e-19
3,8.640000e-15,3.750000e-15,4.935000e-18,-1.432000e-15,-1.857000e-18,3.724000e-16,1.438000e-15,-4.085000e-18,3.724000e-16,8.164000e-18,...,9.211000e-15,5.598000e-15,3.755000e-16,6.000000e-07,0.9351,0.0,0.025,0.0,1.805000e-19,1.176000e-17
4,8.003000e-15,3.559000e-15,2.198000e-18,-1.304000e-15,-8.401000e-19,3.724000e-16,1.307000e-15,-2.247000e-18,3.724000e-16,3.573000e-18,...,5.777000e-16,3.011000e-16,2.293000e-17,6.000000e-07,1.0250,-0.2,0.025,0.0,1.135000e-20,2.892000e-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3376816,4.632000e-15,3.559000e-15,3.960000e-17,8.399000e-15,-1.600000e-17,3.724000e-16,2.583000e-14,-3.421000e-14,3.724000e-16,3.249000e-14,...,1.275000e-04,1.248000e-04,1.938000e-03,3.000000e-06,0.9223,-0.2,9.975,10.0,3.917000e-09,2.522000e-12
3376817,4.601000e-15,3.421000e-15,4.427000e-17,6.646000e-15,-1.845000e-17,3.724000e-16,2.562000e-14,-3.225000e-14,3.724000e-16,3.153000e-14,...,1.336000e-04,1.111000e-04,1.915000e-03,3.000000e-06,1.0190,-0.4,9.975,10.0,3.933000e-09,2.502000e-12
3376818,4.662000e-15,3.750000e-15,3.300000e-17,1.108000e-14,-1.281000e-17,3.724000e-16,2.608000e-14,-3.715000e-14,3.724000e-16,3.377000e-14,...,1.209000e-04,1.447000e-04,1.965000e-03,3.000000e-06,0.8156,0.0,10.000,10.0,3.900000e-09,2.545000e-12
3376819,4.629000e-15,3.559000e-15,3.851000e-17,8.400000e-15,-1.556000e-17,3.724000e-16,2.583000e-14,-3.421000e-14,3.724000e-16,3.249000e-14,...,1.275000e-04,1.248000e-04,1.938000e-03,3.000000e-06,0.9223,-0.2,10.000,10.0,3.918000e-09,2.522000e-12


In [4]:
# sweep variable vectors
l =   np.round(np.unique(df['l'])*1e6, 2)
vgs = np.unique(df['vg'])
vds = np.unique(df['vd'])
vsb = np.unique(-df['vb'])

In [5]:
# ngspice sweep order is l, vgs, vds, vsb
dims = [len(l), len(vgs), len(vds), len(vsb)]
id = np.reshape(df['id'].values, dims)
vt = np.reshape(df['vth'].values, dims)
gm = np.reshape(df['gm'].values, dims)
gmb = np.reshape(df['gmbs'].values, dims)
gds = np.reshape(df['gds'].values, dims)
cgg = np.reshape(df['cgg'].values, dims) \
      + np.reshape(df['cgdo'].values, dims) + np.reshape(df['cgso'].values, dims)
cgb = -np.reshape(df['cgb'].values, dims)
cgd = -np.reshape(df['cgd'].values, dims) \
      + np.reshape(df['cgdo'].values, dims)
cgs = -np.reshape(df['cgs'].values, dims) \
      + np.reshape(df['cgso'].values, dims)
cdd = np.reshape(df['cdd'].values, dims) \
      + np.reshape(df['capbd'].values, dims) \
      + np.reshape(df['cgdo'].values, dims)
css = np.reshape(df['css'].values, dims) \
      + np.reshape(df['capbs'].values, dims) \
      + np.reshape(df['cgso'].values, dims)
sth = np.reshape(df['nid'].values, dims)**2
sfl = np.reshape(df['n1overf'].values, dims)**2


In [6]:
dic = {
  "INFO": "GlobalFoundries, 180nm MCU CMOS , BSIM4",
  "CORNER": "NOM",
  "TEMP": 300.0,
  "VGS": vgs,
  "VDS": vds,
  "VSB": vsb,
  "L": l,
  "W": w[choice],
  "NFING": nfing[choice],
  "ID": id,
  "VT": vt,
  "GM": gm,
  "GMB": gmb,
  "GDS": gds,
  "CGG": cgg,
  "CGB": cgb,
  "CGD": cgd,
  "CGS": cgs,
  "CDD": cdd,
  "CSS": css,
  "STH": sth,
  "SFL": sfl
}
savemat('./simulation/'+devices[choice]+'.mat', {devices[choice]: dic})